# Chapter 51: Introduction to Multimodal AI
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Memahami prinsip dasar integrasi data multimodal berpusat pada visi komputer (Vision-Centric Multimodal Perception), membedah tiga paradigma fusi (Early, Intermediate, Late Fusion), mengimplementasikan pembobotan sadar-keandalan sensor (*uncertainty-aware weighting*), serta mengevaluasi ketahanan sistem terhadap derau sensorik dan sensor dropout.

### 1. Landasan Teori & Formulasi Matematis Fusi Multimodal
Fusi Late dengan Pembobotan Keandalan Sensor:
$$\mathbf{p}_{\text{fused}} = \sum_{m=1}^M \alpha_m \mathbf{p}_m, \quad \alpha_m = \frac{\gamma_m}{\sum_{k=1}^M \gamma_k}$$
Di mana $\gamma_m$ merupakan koefisien keandalan sensorik (misalnya invers varians atau $1 - \text{Entropy}/\log C$).

Early Fusion Concatenation:
$$\mathbf{z} = \frac{[\mathbf{v}^T, \mathbf{a}^T, \mathbf{s}^T]^T}{\|[\mathbf{v}^T, \mathbf{a}^T, \mathbf{s}^T]^T\|_2 + \epsilon}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Fusi Multimodal & Validasi Contoh Analitik Seksi 3
def l2_norm(vec, eps=1e-12):
    n = np.linalg.norm(vec)
    return vec / max(n, eps)

def late_fusion(probs_list, rel_list):
    probs = np.array(probs_list, dtype=np.float64)
    rels = np.array(rel_list, dtype=np.float64)
    tot = np.sum(rels)
    weights = rels / tot if tot > 0 else np.ones_like(rels) / len(rels)
    fused = np.sum(weights[:, np.newaxis] * probs, axis=0)
    return fused, weights

# Data Pengamatan 3 Modalitas: Visual (v), Audio (a), Radar (r)
# 3 Kelas: [0: Kendaraan, 1: Pejalan Kaki, 2: Derau]
p_v = [0.70, 0.20, 0.10]
p_a = [0.40, 0.50, 0.10]
p_r = [0.85, 0.10, 0.05]

gamma = [0.80, 0.40, 0.80]  # Audio bising -> gamma_a = 0.40

p_fused, alpha = late_fusion([p_v, p_a, p_r], gamma)
print(f"Bobot Terkalibrasi: alpha_v={alpha[0]:.2f}, alpha_a={alpha[1]:.2f}, alpha_r={alpha[2]:.2f}")
print(f"Probabilitas Fusi: Class 0={p_fused[0]:.4f}, Class 1={p_fused[1]:.4f}, Class 2={p_fused[2]:.4f}")

# Validasi eksak dengan Seksi 3
np.testing.assert_allclose(alpha, [0.40, 0.20, 0.40], atol=1e-5)
np.testing.assert_allclose(p_fused, [0.7000, 0.2200, 0.0800], atol=1e-5)
print("VALIDASI BERHASIL: Hasil komputasi late fusion cocok sempurna dengan derivasi matematis!")

In [ ]:
# 4. Evaluasi Ketahanan Derau & Simulasi Sensor Dropout
# Fusi Naif Rata-Rata (Unweighted)
p_naive, _ = late_fusion([p_v, p_a, p_r], [1.0, 1.0, 1.0])
print(f"Fusi Naif Rata-Rata: Class 0={p_naive[0]:.4f}")
assert p_fused[0] > p_naive[0], "Fusi sadar-keandalan harus lebih percaya diri daripada fusi naif!"
print(f"Peningkatan Keyakinan: +{(p_fused[0] - p_naive[0])*100:.2f}%")

# Simulasi Sensor Dropout: Kamera Tertutup (gamma_v = 0.0)
p_drop, alpha_drop = late_fusion([p_v, p_a, p_r], [0.0, 0.40, 0.80])
print(f"\n--- Skenario Kamera Dropout (gamma_v = 0) ---")
print(f"Bobot Fallback: Audio={alpha_drop[1]:.4f}, Radar={alpha_drop[2]:.4f}")
print(f"Probabilitas Fallback: Class 0={p_drop[0]:.4f}, Class 1={p_drop[1]:.4f}")
np.testing.assert_allclose(p_drop[0], 0.7000, atol=1e-4)
print("VERIFIKASI FAIL-SAFE DROPOUT: Lulus deterministik.")

In [ ]:
# 5. Spesifikasi Modul Fusi PyTorch (Untuk GPU / Google Colab)
if HAS_TORCH:
    print("Mode PyTorch Aktif.")
    class LateFusionModule(nn.Module):
        def __init__(self, num_modalities=3, num_classes=3):
            super().__init__()
            self.weight_logits = nn.Parameter(torch.ones(num_modalities))
        def forward(self, probs):
            # probs: (B, M, C)
            weights = torch.softmax(self.weight_logits, dim=0)
            return torch.sum(probs * weights.view(1, -1, 1), dim=1)
    model = LateFusionModule()
    print("Inisialisasi LateFusionModule PyTorch berhasil.")
else:
    print("PANDUAN EKSEKUSI TRAINING MULTIMODAL DI GOOGLE COLAB / GPU:")
    print("1. Untuk melatih arsitektur fusi multimodal end-to-end:")
    print("   !pip install torch torchvision torchaudio transformers")
    print("2. Arsitektur direkomendasikan:")
    print("   - Visual Backbone: ViT-B/16 atau ResNet-50")
    print("   - Audio Backbone: Audio Spectrogram Transformer (AST)")
    print("   - Fusion Layer: Gated Cross-Attention atau Multimodal Bottleneck Transformer (MBT)")
    print("   - Loss: Multimodal Focal Loss / Uncertainty-regularized Cross-Entropy")

### 6. Rangkuman & Pembahasan Akademik
1. **Vision-Centric Anchor**: Memosisikan visi komputer sebagai modalitas inti, sementara audio dan sensor radar bertindak sebagai penguat persepsi yang handal.
2. **Pentingnya Kalibrasi Keandalan**: Fusi naif seragam terbukti menurunkan performa saat salah satu sensor terdegradasi; pembobotan berbasis varians atau entropi mencegah *negative transfer*.
3. **Desain Fail-Safe**: Sistem multimodal yang tangguh mampu beradaptasi secara dinamis terhadap sensor dropout tanpa mengalami kegagalan fatal.